# Week 4 Day 3 - create_agent、agentレイヤー

昨日は、LangGraphでtool loopを手作業で組み立てました。chatbotノード、toolsノード、条件分岐エッジ、そして戻るためのエッジです。今日は、それらすべてを1回の関数呼び出しで手に入れます。

`create_agent`はレイヤー3です。モデルと、いくつかのツール、そしてプロンプトを渡すだけで、agent loopを構築してくれます。これは、制御から便利さへと向かう道筋の次のステップです。ループの制御をフレームワークに任せることで、書くコードがはるかに少なくなります。この週全体をつなぐ大事な点は、それが構築するものが実はLangGraphのグラフであり、昨日手作業で組み立てたものと同じ種類のものだということです。agentにツールを持たせたら、それを実際に確認していきます。

In [ ]:
# まずはインポートと環境設定を、すべてまとめて

from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv(override=True)

## パート1: 最もシンプルなagent

最も単純な形のagentは、promptを持つモデルです。モデルは`provider:model`という形式の文字列として渡し、その性格を設定するためのsystem promptも渡します。

In [ ]:
agent = create_agent(
    model="openai:gpt-5.4-mini",
    system_prompt="You are a helpful assistant who answers concisely.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "What is the Model Context Protocol, in two sentences?"}]})
print(result["messages"][-1].content)

### そして非同期版の相棒: ainvoke

LangChainで`invoke`できるものは、すべて`ainvoke`もできます。引数も結果も同じで、awaitするだけです。asyncioについてはコースの前半でよく学んでいるので、新しく学ぶことは何もありません。これは単に、非同期のコードからagentを呼び出すための正しい方法です。このラボの最後、ブラウザのツールが非同期しか話さない場面で、これが重要になります。

In [ ]:
result = await agent.ainvoke({"messages": [{"role": "user", "content": "In one sentence: why does async code suit agents so well?"}]})
print(result["messages"][-1].content)

## パート2: ツール

agentを実用的にするために、ツールを与えます。これらはDay 1と同じ`@tool`関数です。リストとして渡すだけで、agentがtool loop全体を実行してくれます。

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Return today's weather for a city."""
    pretend = {"London": "rainy, 14 degrees", "Rome": "sunny, 27 degrees"}
    return pretend.get(city, "clear, 20 degrees")

@tool
def get_population(city: str) -> str:
    """Return the population of a city."""
    pretend = {"London": "8.9 million", "Rome": "2.8 million"}
    return pretend.get(city, "unknown")

agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[get_weather, get_population],
    system_prompt="You are a travel assistant. Use your tools to answer questions about cities.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "What is the weather and population of Rome?"}]})
print(result["messages"][-1].content)

## 種明かし: これはLangGraphのグラフです

agentにツールを持たせたので、それを描画してみましょう。`create_agent`はコンパイル済みのLangGraphグラフを返すので、昨日使ったのとまったく同じ呼び出しで描画できます。その形を見てください。modelノード、toolsノード、そしてその間の条件分岐によるループです。これは、Day 2に手作業で組み立てたグラフが、たった1行で手に入ったということです。

In [ ]:
display(Image(agent.get_graph().draw_mermaid_png()))

## パート3: メモリ

デフォルトでは、`invoke`の呼び出しはそれぞれ新しい始まりになります。会話を記憶させるには、昨日グラフに対して行ったのと同じように、agentにcheckpointerを与え、どの会話を指しているのかを知らせるために`thread_id`を渡します。

In [ ]:
memory_agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[get_weather],
    checkpointer=MemorySaver(),
    system_prompt="You are a travel assistant. Use your tools to answer questions about cities.",
)

config = {"configurable": {"thread_id": "trip-planning"}}
memory_agent.invoke({"messages": [{"role": "user", "content": "I am planning a trip to London."}]}, config=config)
result = memory_agent.invoke({"messages": [{"role": "user", "content": "What is the weather like where I am going on my trip?"}]}, config=config)
print(result["messages"][-1].content)

## パート4: structured output

文章ではなく型付きのオブジェクトを返してほしいときは、Pydanticモデルを`response_format`として渡します。agentはそれでもきちんと作業を行い、ツールを使いますが、最後に自分のオブジェクトに値を埋めてくれます。それは`structured_response`キーから読み取ります。

In [ ]:
class CityReport(BaseModel):
    city: str = Field(description="The city name")
    weather: str = Field(description="A short weather description")
    population: str = Field(description="The population")

report_agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[get_weather, get_population],
    response_format=CityReport,
)

result = report_agent.invoke({"messages": [{"role": "user", "content": "Give me a report on London."}]})
report = result["structured_response"]
print(report)
print("Just the weather:", report.weather)

## パート5: ミドルウェア

ミドルウェアは、agentの挙動を形作るための強力な機能です。これによって、ループの中の決まったポイントで、自分自身のコードを実行できます。モデルが呼び出される前、モデルが答えたあと、あるいは各ツール呼び出しの前後などです。

ここでは、ツール呼び出しが発生するたびにそれを表示する、小さな独自ミドルウェアを紹介します。これで、agentが作業している様子を観察できます。`@wrap_tool_call`デコレータは各ツール呼び出しをラップします。私たちはそれをログに記録し、続けて`handler`を呼び出して処理を進めます。

In [ ]:
@wrap_tool_call
def log_tool_calls(request, handler):
    call = request.tool_call
    print(f"  [middleware] calling {call['name']} with {call['args']}")
    return handler(request)

watched_agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[get_weather, get_population],
    system_prompt="You are a travel assistant. Use your tools.",
    middleware=[log_tool_calls],
)

result = watched_agent.invoke({"messages": [{"role": "user", "content": "Weather and population of London and Rome?"}]})
print("\nFinal answer:", result["messages"][-1].content)

LangChainには、あらかじめ用意された様々なミドルウェアも用意されています。長い会話をcontext window内に収めるための`SummarizationMiddleware`、機密データを伏せ字にする`PIIMiddleware`、リトライや呼び出し回数の制限を行うミドルウェア、そして人による承認を待って処理を一旦止める`HumanInTheLoopMiddleware`などです。

## パート6の前に: NodeとPlaywright

このラボの最後の部分では、別のプログラムであるMCPサーバー上に存在するツールを使います。そのプログラムはNode上で動作します。始める前に、2つ簡単に確認しておきましょう。

まずはNode自体です。v22以降が必要です。以下のセルが失敗する場合は、次のいずれかのコマンドでNodeをインストールしてください。

- **Windows**の場合、PowerShellで: `winget install OpenJS.NodeJS.LTS`
- **Mac**の場合、Terminalで: `brew install node`
- **Linux**、またはどちらもうまくいかない場合は、[setup/SETUP-node.md](../setup/SETUP-node.md)のガイドを参照してください

インストール後は、Cursorを完全に終了してから再度起動し、このノートブックを開き直して、ラボを最初からやり直してください。この完全な再起動は重要です。新しくインストールしたNodeは、すでに動作しているノートブックからは見えず、カーネルだけを再起動しても不十分です。

In [ ]:
!node --version
!npx --version

次はPlaywrightです。Microsoftのブラウザ自動化フレームワークです。インストールは何も必要ありません。`npx`が必要に応じて取得し、すでにマシンにあるChromeを操作します。Chromeが起動している必要はありません。Playwrightが自分でChromeを起動します。

以下のセルは、AIをまったく介さずに、この一連の仕組み全体を実証します。NodeがPlaywrightを実行し、PlaywrightがChromeを開き、Hacker Newsを読み込み、スクリーンショットを保存します。npxがパッケージをダウンロードするため、最初の実行だけ少し時間がかかります。

Chromeが見つからないというエラーが出る場合は、通常の方法でChromeをインストールするか、ターミナルで`npx playwright install chrome`を実行してから、もう一度試してください。

In [ ]:
!npx -y playwright@latest screenshot --channel=chrome https://news.ycombinator.com playwright_check.png
display(Image("playwright_check.png"))

## パート6: MCPサーバーと、本物のブラウザ

ツールは、自分で書くPython関数である必要はありません。Model Context Protocolは、別のサーバー上に存在するツールをagentが使うための標準的な方法です。LangChainは、`langchain-mcp-adapters`を使ってそれらのツールを読み込み、agentからすれば、それらは他のツールと変わらないただのツールです。

ここでは、Microsoftが提供するPlaywright MCPサーバーに接続します。これは、先ほど動作確認したのと同じ本物のブラウザを操作します。デフォルトではウィンドウを表示するモード(headed)で動くので、動いている様子を見ることができます。ディスプレイのないLinuxマシンでは、静かにheadlessモードにフォールバックします。これらのツールは別プロセスと通信するため非同期(async)です。そのため`await`で読み込み、パート1で紹介した`ainvoke`でagentを実行します。

### Windows向けの1つの調整

Pythonが別のプログラムを起動するとき、そのプログラムにエラーメッセージを書き込む場所を渡します。Windows上のJupyterノートブックの中では、その場所が実際のファイルではないため、MCPサーバーを起動するライブラリがそこでつまずいてしまいます（これはMCP Python SDKの既知の問題で、ノートブックの中でしか現れず、通常のPythonスクリプトでは発生しません）。以下のセルは、接続する前にMCPサーバーのエラーログを安全な場所に向けます。MacとLinuxでは何も行わないので、どちらの環境でも実行して先に進んでください。

In [ ]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

In [ ]:
client = MultiServerMCPClient({
    "playwright": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@playwright/mcp@latest", "--isolated"],
    }
})

browser_tools = await client.get_tools()
print(f"Loaded {len(browser_tools)} browser tools:")
for t in browser_tools:
    print(" -", t.name)

それでは、これらのブラウザツールを普通の`create_agent`に渡し、Hacker Newsにアクセスして見つけたことを報告するよう頼んでみましょう。ブラウザのウィンドウが自分で開き、自分でナビゲートしていく様子を見てみてください。

In [ ]:
browser_agent = create_agent(
    model="openai:gpt-5.5",
    tools=browser_tools,
    system_prompt="You are a web research assistant. Use the browser tools to complete the task, then report clearly.",
)

result = await browser_agent.ainvoke({"messages": [{"role": "user",
    "content": "Go to https://news.ycombinator.com and tell me the titles of the top three stories on the front page."}]})
print(result["messages"][-1].content)

## まとめ、そしてこれから向かう先

このラボ1つの中で、たった1行でagentを作り、それが実はLangGraphのグラフであることを確認し、ツール、メモリ、structured outputを与え、ミドルウェアでその挙動を形作り、MCPサーバーを通じて本物のブラウザを操作させました。これは、あなた自身の仕事の中で最もよく使うことになるレイヤーです。

明日は、さらに1段階上のレイヤーに進みます。Deep Agentsは、create_agentを取り込み、多くのステップを必要とする作業のための、本格的なハーネスの中に包み込みます。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">ブラウザagentにcheckpointerとthread_idを与え、それと短いやり取りをしてみましょう。あるサイトを開くよう頼み、その後、見つけた内容についての追加の質問をして、agentがそれを覚えているか確認してください。もっと難しい課題に挑戦したい場合は、自分がブロックリストに載せたサイトへのナビゲーションを拒否するミドルウェアを書き、それがきちんと機能することを確認してみてください。
            </span>
        </td>
    </tr>
</table>